In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import matplotlib as mpl
from sklearn.model_selection import train_test_split, cross_val_score, RepeatedStratifiedKFold, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.feature_selection import RFE
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, classification_report, 
                            roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
                            brier_score_loss, f1_score, accuracy_score, precision_score, recall_score)
from sklearn.calibration import calibration_curve
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import joblib
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.feature_selection import SelectKBest, f_classif
from numpy import mean, std

# ===============================================================
# CREATE OUTPUT DIRECTORIES FIRST
# ===============================================================
Path("Amodels").mkdir(exist_ok=True)
Path("Afigures").mkdir(exist_ok=True)
Path("Aresults").mkdir(exist_ok=True)
Path("Arfe_results").mkdir(exist_ok=True)

# ===============================================================
# GLOBAL SETTINGS & STYLES
# ===============================================================
plt.rcParams.update({
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.linewidth": 0.6,
    "grid.alpha": 0.35,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Arial", "Helvetica"],
    "figure.dpi": 150,
})

label_size = 14
mpl.rcParams['xtick.labelsize'] = label_size - 2
mpl.rcParams['ytick.labelsize'] = label_size - 2
mpl.rcParams['axes.labelsize'] = label_size
mpl.rcParams['axes.titlesize'] = label_size + 2
mpl.rcParams['legend.fontsize'] = label_size - 2
mpl.rcParams['figure.titlesize'] = label_size + 4

# Color palette
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#6A994E', '#8B5FBF', '#0F7173']
sns.set_palette(sns.color_palette(colors))

# Suppress warnings
warnings.filterwarnings('ignore')

# ===============================================================
# UTILITY FUNCTIONS
# ===============================================================
def save_figure(prefix, fig=None, dpi=300, results_dir="figures"):
    """Save figure in multiple formats"""
    results_dir = Path(results_dir)
    results_dir.mkdir(exist_ok=True)
    if fig is None:
        fig = plt.gcf()
    for ext in ["pdf", "png"]:
        out = results_dir / f"{prefix}.{ext}"
        fig.savefig(
            out,
            bbox_inches="tight",
            pad_inches=0.1,
            dpi=dpi if ext == "png" else None,
            transparent=True
        )
    print(f"[SAVED] {prefix}.pdf/.png")

def create_classification_report_plot(y_true, y_pred, model_name):
    """Create visual classification report"""
    report = classification_report(y_true, y_pred, output_dict=True)
    metrics_df = pd.DataFrame(report).transpose()
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.axis('tight')
    ax.axis('off')
    
    table = ax.table(cellText=np.round(metrics_df.values, 3),
                     rowLabels=metrics_df.index,
                     colLabels=metrics_df.columns,
                     cellLoc='center',
                     loc='center')
    
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1.2, 1.5)
    
    plt.title(f'Classification Report - {model_name}', fontsize=16, pad=20)
    return fig

def plot_upper_triangular_correlation_matrix(data, title="Correlation Matrix", figsize=(12, 10)):
    """Plot upper triangular correlation matrix"""
    corr_matrix = data.corr()
    
    # Create mask for upper triangle
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    
    fig, ax = plt.subplots(figsize=figsize)
    cmap = sns.diverging_palette(230, 20, as_cmap=True)
    
    # Plot heatmap
    sns.heatmap(corr_matrix, mask=mask, cmap=cmap, vmax=1, vmin=-1, center=0,
                square=True, linewidths=0.5, cbar_kws={"shrink": 0.8}, annot=False,
                ax=ax)
    
    # Add annotations for significant correlations
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):  # Upper triangle only
            corr_value = corr_matrix.iloc[i, j]
            if abs(corr_value) > 0.5:  # Only show strong correlations
                ax.text(j + 0.5, i + 0.5, f'{corr_value:.2f}', 
                       ha='center', va='center', fontsize=9, fontweight='bold')
    
    ax.set_title(title, fontsize=16, pad=20)
    plt.tight_layout()
    return fig

# ===============================================================
# DATA LOADING & PREPROCESSING
# ===============================================================
print("="*50)
print("DATA LOADING & PREPROCESSING")
print("="*50)

# Load data
df = pd.read_csv('cybersecurity_intrusion_data.csv')

# Define features
numeric_features = ['network_packet_size', 'login_attempts', 'session_duration', 
                    'ip_reputation_score', 'failed_logins']
binary_features = ['unusual_time_access']
categorical_features = ['protocol_type', 'encryption_used', 'browser_type']
target = 'attack_detected'

X = df.drop(columns=[target, 'session_id'])
y = df[target]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Class distribution (Train): {np.bincount(y_train)}")
print(f"Class distribution (Test): {np.bincount(y_test)}")

# ===============================================================
# PREPROCESSING PIPELINE
# ===============================================================
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
    ('bin', 'passthrough', binary_features)
])

# ===============================================================
# 1. BASELINE LOGISTIC REGRESSION MODEL (WITH SMOTE)
# ===============================================================
print("\n" + "="*50)
print("1. BASELINE LOGISTIC REGRESSION MODEL")
print("="*50)

logistic_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=42,
        solver='liblinear'
    ))
])

print("Training Baseline Logistic Regression...")
logistic_pipeline.fit(X_train, y_train)
y_pred_lr = logistic_pipeline.predict(X_test)
y_proba_lr = logistic_pipeline.predict_proba(X_test)[:, 1]

# Save model
joblib.dump(logistic_pipeline, 'Amodels/logistic_regression_model.pkl')
print("✓ Baseline Logistic Regression model saved")

# ===============================================================
# 2. RANDOM FOREST MODEL (WITH SMOTE)
# ===============================================================
print("\n" + "="*50)
print("2. RANDOM FOREST MODEL")
print("="*50)

rf_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

print("Training Random Forest...")
rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)
y_proba_rf = rf_pipeline.predict_proba(X_test)[:, 1]

# Save model
joblib.dump(rf_pipeline, 'Amodels/random_forest_model.pkl')
print("✓ Random Forest model saved")

# ===============================================================
# 3. HYBRID MODEL (STACKING CLASSIFIER)
# ===============================================================
print("\n" + "="*50)
print("3. HYBRID MODEL (Stacking Classifier)")
print("="*50)

base_estimators = [
    ('logistic', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=42,
        solver='liblinear'
    )),
    ('rf', RandomForestClassifier(
        n_estimators=50,
        max_depth=8,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
]

stacking_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', StackingClassifier(
        estimators=base_estimators,
        final_estimator=LogisticRegression(
            class_weight='balanced',
            random_state=42,
            max_iter=1000
        ),
        cv=5,
        n_jobs=-1
    ))
])

print("Training Stacking Classifier...")
stacking_pipeline.fit(X_train, y_train)
y_pred_stack = stacking_pipeline.predict(X_test)
y_proba_stack = stacking_pipeline.predict_proba(X_test)[:, 1]

joblib.dump(stacking_pipeline, 'Amodels/stacking_model.pkl')
print("✓ Stacking model saved")

# ===============================================================
# 4. RECURSIVE FEATURE ELIMINATION (RFE) ANALYSIS
# ===============================================================
print("\n" + "="*50)
print("4. RECURSIVE FEATURE ELIMINATION (RFE) ANALYSIS")
print("="*50)

# First, let's preprocess the data to get feature names
preprocessor.fit(X_train)
X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

# Get feature names after preprocessing
feature_names = preprocessor.get_feature_names_out()

print(f"Number of features after preprocessing: {len(feature_names)}")

# Convert to DataFrame for easier handling
X_train_df = pd.DataFrame(X_train_transformed, columns=feature_names)
X_test_df = pd.DataFrame(X_test_transformed, columns=feature_names)

# ===============================================================
# 4.1 RFE with Random Forest - Finding Optimal Number of Features
# ===============================================================
print("\nPerforming RFE to find optimal number of features...")

def evaluate_rfe_models(X, y, base_model, n_features_range=range(2, 15)):
    """Evaluate RFE with different numbers of features"""
    results = {}
    
    for n_features in n_features_range:
        # Create RFE with base model
        rfe = RFE(estimator=base_model, n_features_to_select=n_features)
        
        # Create pipeline with RFE and classifier
        model = RandomForestClassifier(
            n_estimators=50,
            max_depth=5,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        )
        
        pipeline = Pipeline(steps=[('s', rfe), ('m', model)])
        
        # Evaluate using cross-validation
        cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=42)
        scores = cross_val_score(pipeline, X, y, scoring='roc_auc', cv=cv, n_jobs=-1)
        
        results[n_features] = {
            'mean_score': mean(scores),
            'std_score': std(scores),
            'scores': scores
        }
        
        print(f'> {n_features} features: ROC-AUC = {mean(scores):.3f} (±{std(scores):.3f})')
    
    return results

# Evaluate RFE with Random Forest
rfe_results = evaluate_rfe_models(X_train_df, y_train, 
                                 RandomForestClassifier(n_estimators=50, random_state=42),
                                 n_features_range=range(2, min(20, len(feature_names))))

# Find optimal number of features
best_n_features = max(rfe_results.items(), key=lambda x: x[1]['mean_score'])[0]
print(f"\nOptimal number of features: {best_n_features}")

# ===============================================================
# 4.2 Plot RFE Results
# ===============================================================
print("\nPlotting RFE results...")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Box plot
n_features_list = list(rfe_results.keys())
scores_list = [rfe_results[n]['scores'] for n in n_features_list]

ax1.boxplot(scores_list, labels=[str(n) for n in n_features_list], showmeans=True)
ax1.set_xlabel('Number of Features Selected', fontsize=12)
ax1.set_ylabel('ROC-AUC Score', fontsize=12)
ax1.set_title('RFE: Model Performance vs Number of Features', fontsize=14, pad=15)
ax1.grid(True, alpha=0.3, linestyle='--')

# Line plot with error bars
means = [rfe_results[n]['mean_score'] for n in n_features_list]
stds = [rfe_results[n]['std_score'] for n in n_features_list]

ax2.errorbar(n_features_list, means, yerr=stds, fmt='-o', capsize=5, capthick=2, linewidth=2)
ax2.axvline(x=best_n_features, color='r', linestyle='--', alpha=0.7, label=f'Best: {best_n_features} features')
ax2.set_xlabel('Number of Features Selected', fontsize=12)
ax2.set_ylabel('Mean ROC-AUC Score', fontsize=12)
ax2.set_title('RFE: Mean Performance with Confidence Intervals', fontsize=14, pad=15)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
save_figure('rfe_feature_selection_performance', fig)
plt.close(fig)

# ===============================================================
# 4.3 Train Final Model with RFE Selected Features
# ===============================================================
print(f"\nTraining final model with {best_n_features} selected features...")

# Create RFE with optimal number of features
rfe = RFE(
    estimator=RandomForestClassifier(n_estimators=50, random_state=42),
    n_features_to_select=best_n_features
)

# Create pipeline with RFE
rfe_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('rfe', rfe),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

# Train RFE model
rfe_pipeline.fit(X_train, y_train)

# Get selected features
rfe_model = rfe_pipeline.named_steps['rfe']
selected_features_mask = rfe_model.support_

# Get selected feature names
selected_feature_names = feature_names[selected_features_mask]
print(f"\nSelected {len(selected_feature_names)} features:")
for i, feature in enumerate(selected_feature_names, 1):
    print(f"  {i}. {feature}")

# Make predictions with RFE model
y_pred_rfe = rfe_pipeline.predict(X_test)
y_proba_rfe = rfe_pipeline.predict_proba(X_test)[:, 1]

# Save RFE model
joblib.dump(rfe_pipeline, 'Amodels/rfe_model.pkl')
print("✓ RFE model saved")

# Save selected features
selected_features_df = pd.DataFrame({
    'feature_name': selected_feature_names,
    'feature_rank': rfe_model.ranking_[selected_features_mask]
}).sort_values('feature_rank')

selected_features_df.to_csv('Arfe_results/selected_features.csv', index=False)

# ===============================================================
# 4.4 Upper Triangular Correlation Matrix of Selected Features
# ===============================================================
print("\nGenerating correlation matrix of selected features...")

# Get the transformed data with only selected features
X_train_selected = X_train_df[selected_feature_names]

# Plot correlation matrix
fig = plot_upper_triangular_correlation_matrix(
    X_train_selected,
    title=f'Correlation Matrix of Selected Features ({len(selected_feature_names)} features)',
    figsize=(12, 10)
)
save_figure('selected_features_correlation_matrix', fig)
plt.close(fig)

# ===============================================================
# 5. COMPARISON VISUALIZATIONS: WITH VS WITHOUT FEATURE SELECTION
# ===============================================================
print("\n" + "="*50)
print("5. COMPARISON VISUALIZATIONS")
print("="*50)

# ===============================================================
# 5.1 Calibration Curves Comparison
# ===============================================================
print("Generating calibration curves comparison...")
print("\nTraining RFE versions of all models for comparison...")

# RFE Logistic Regression
rfe_lr_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('rfe', RFE(
        estimator=LogisticRegression(random_state=42),
        n_features_to_select=best_n_features
    )),
    ('classifier', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=42,
        solver='liblinear'
    ))
])
rfe_lr_pipeline.fit(X_train, y_train)
y_proba_rfe_lr = rfe_lr_pipeline.predict_proba(X_test)[:, 1]

# RFE Stacking (simplified version)
rfe_stacking_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('rfe', RFE(
        estimator=RandomForestClassifier(n_estimators=50, random_state=42),
        n_features_to_select=best_n_features
    )),
    ('classifier', StackingClassifier(
        estimators=[
            ('logistic', LogisticRegression(class_weight='balanced', random_state=42)),
            ('rf', RandomForestClassifier(n_estimators=50, class_weight='balanced', random_state=42))
        ],
        final_estimator=LogisticRegression(class_weight='balanced', random_state=42),
        cv=3
    ))
])
rfe_stacking_pipeline.fit(X_train, y_train)
y_proba_rfe_stack = rfe_stacking_pipeline.predict_proba(X_test)[:, 1]

# Create model pairs for comparison
model_pairs = [
    (y_proba_lr, 'Logistic Regression (Full)', y_proba_rfe_lr, 'Logistic Regression (RFE)'),
    (y_proba_rf, 'Random Forest (Full)', y_proba_rfe, 'Random Forest (RFE)'),
    (y_proba_stack, 'Stacking (Full)', y_proba_rfe_stack, 'Stacking (RFE)'),
]

# Create calibration curves with 1x3 layout
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Calibration Curves: With vs Without Feature Selection', fontsize=18, y=1.02)

for ax, (full_proba, full_label, rfe_proba, rfe_label) in zip(axes, model_pairs):
    # Full model calibration
    prob_true_full, prob_pred_full = calibration_curve(y_test, full_proba, n_bins=10, strategy='uniform')
    brier_full = brier_score_loss(y_test, full_proba)
    
    # RFE model calibration
    prob_true_rfe, prob_pred_rfe = calibration_curve(y_test, rfe_proba, n_bins=10, strategy='uniform')
    brier_rfe = brier_score_loss(y_test, rfe_proba)
    
    # Plot both curves
    ax.plot(prob_pred_full, prob_true_full, 's-', label=f'{full_label}\nBrier: {brier_full:.4f}', 
            linewidth=2, markersize=8)
    ax.plot(prob_pred_rfe, prob_true_rfe, 'o-', label=f'{rfe_label}\nBrier: {brier_rfe:.4f}', 
            linewidth=2, markersize=8)
    ax.plot([0, 1], [0, 1], 'k--', label='Perfectly calibrated', alpha=0.5)
    
    ax.set_xlabel('Mean Predicted Probability', fontsize=12)
    ax.set_ylabel('Fraction of Positives', fontsize=12)
    ax.set_title(f'{full_label.split(" (")[0]} Calibration', fontsize=14)
    ax.legend(loc='upper left', fontsize=10)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])

plt.tight_layout()
save_figure('calibration_curves_comparison_rfe_vs_full', fig)
plt.close(fig)

# ===============================================================
# 5.2 ROC Curves Comparison
# ===============================================================
print("Generating ROC curves comparison...")
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('ROC Curves: With vs Without Feature Selection', fontsize=18, y=1.02)

for ax, (full_proba, full_label, rfe_proba, rfe_label) in zip(axes, model_pairs):
    # Full model ROC
    fpr_full, tpr_full, _ = roc_curve(y_test, full_proba)
    roc_auc_full = roc_auc_score(y_test, full_proba)
    
    # RFE model ROC
    fpr_rfe, tpr_rfe, _ = roc_curve(y_test, rfe_proba)
    roc_auc_rfe = roc_auc_score(y_test, rfe_proba)
    
    # Plot both curves
    ax.plot(fpr_full, tpr_full, 'b-', label=f'{full_label}\nAUC = {roc_auc_full:.3f}', 
            linewidth=2.5, alpha=0.8)
    ax.plot(fpr_rfe, tpr_rfe, 'r-', label=f'{rfe_label}\nAUC = {roc_auc_rfe:.3f}', 
            linewidth=2.5, alpha=0.8)
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.6)
    
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title(f'{full_label.split(" (")[0]} ROC Curves', fontsize=14)
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])

plt.tight_layout()
save_figure('roc_curves_comparison_rfe_vs_full', fig)
plt.close(fig)

# ===============================================================
# 5.3 Precision-Recall Curves Comparison
# ===============================================================
print("Generating Precision-Recall curves comparison...")
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Precision-Recall Curves: With vs Without Feature Selection', fontsize=18, y=1.02)

for ax, (full_proba, full_label, rfe_proba, rfe_label) in zip(axes, model_pairs):
    # Full model PR curve
    precision_full, recall_full, _ = precision_recall_curve(y_test, full_proba)
    avg_precision_full = average_precision_score(y_test, full_proba)
    
    # RFE model PR curve
    precision_rfe, recall_rfe, _ = precision_recall_curve(y_test, rfe_proba)
    avg_precision_rfe = average_precision_score(y_test, rfe_proba)
    
    # Plot both curves
    ax.plot(recall_full, precision_full, 'b-', 
            label=f'{full_label}\nAP = {avg_precision_full:.3f}', 
            linewidth=2.5, alpha=0.8)
    ax.plot(recall_rfe, precision_rfe, 'r-', 
            label=f'{rfe_label}\nAP = {avg_precision_rfe:.3f}', 
            linewidth=2.5, alpha=0.8)
    
    ax.set_xlabel('Recall', fontsize=12)
    ax.set_ylabel('Precision', fontsize=12)
    ax.set_title(f'{full_label.split(" (")[0]} Precision-Recall Curves', fontsize=14)
    ax.legend(loc='upper right', fontsize=10)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])

plt.tight_layout()
save_figure('pr_curves_comparison_rfe_vs_full', fig)
plt.close(fig)

# ===============================================================
# 5.4 Combined Comparison Plot (All models together)
# ===============================================================
print("Generating combined comparison plot...")
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Model Performance: Full Features vs RFE Selected Features', fontsize=18, y=1.05)

# Collect all models for combined plot
all_models = [
    ('Logistic Regression', y_proba_lr, y_proba_rfe_lr),
    ('Random Forest', y_proba_rf, y_proba_rfe),
    ('Stacking', y_proba_stack, y_proba_rfe_stack),
]

# ROC Curve (all together)
ax = axes[0]
for name, full_proba, rfe_proba in all_models:
    fpr_full, tpr_full, _ = roc_curve(y_test, full_proba)
    fpr_rfe, tpr_rfe, _ = roc_curve(y_test, rfe_proba)
    
    ax.plot(fpr_full, tpr_full, '-', label=f'{name} (Full)', linewidth=2, alpha=0.7)
    ax.plot(fpr_rfe, tpr_rfe, '--', label=f'{name} (RFE)', linewidth=2, alpha=0.7)

ax.plot([0, 1], [0, 1], 'k:', alpha=0.5)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves Comparison', fontsize=14)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3, linestyle='--')

# Precision-Recall Curve (all together)
ax = axes[1]
for name, full_proba, rfe_proba in all_models:
    precision_full, recall_full, _ = precision_recall_curve(y_test, full_proba)
    precision_rfe, recall_rfe, _ = precision_recall_curve(y_test, rfe_proba)
    
    ax.plot(recall_full, precision_full, '-', label=f'{name} (Full)', linewidth=2, alpha=0.7)
    ax.plot(recall_rfe, precision_rfe, '--', label=f'{name} (RFE)', linewidth=2, alpha=0.7)

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves Comparison', fontsize=14)
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3, linestyle='--')

# Bar chart of AUC scores
ax = axes[2]
x = np.arange(len(all_models))
width = 0.35

full_auc_scores = [roc_auc_score(y_test, full_proba) for _, full_proba, _ in all_models]
rfe_auc_scores = [roc_auc_score(y_test, rfe_proba) for _, _, rfe_proba in all_models]

ax.bar(x - width/2, full_auc_scores, width, label='Full Features', alpha=0.8)
ax.bar(x + width/2, rfe_auc_scores, width, label='RFE Selected', alpha=0.8)

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('ROC-AUC Score', fontsize=12)
ax.set_title('ROC-AUC Scores Comparison', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels([name for name, _, _ in all_models], fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, linestyle='--', axis='y')

# Add value labels on bars
for i, (full_val, rfe_val) in enumerate(zip(full_auc_scores, rfe_auc_scores)):
    ax.text(i - width/2, full_val + 0.01, f'{full_val:.3f}', 
            ha='center', va='bottom', fontsize=9)
    ax.text(i + width/2, rfe_val + 0.01, f'{rfe_val:.3f}', 
            ha='center', va='bottom', fontsize=9)

plt.tight_layout()
save_figure('combined_comparison_rfe_vs_full', fig)
plt.close(fig)

# ===============================================================
# 6. COMPREHENSIVE MODEL COMPARISON (All Models Including RFE)
# ===============================================================
print("\n" + "="*50)
print("6. COMPREHENSIVE MODEL COMPARISON")
print("="*50)

# Prepare data for comprehensive comparison
all_model_data = [
    ('Logistic (Full)', y_pred_lr, y_proba_lr),
    ('Random Forest (Full)', y_pred_rf, y_proba_rf),
    ('Stacking (Full)', y_pred_stack, y_proba_stack),
    ('Logistic (RFE)', rfe_lr_pipeline.predict(X_test), y_proba_rfe_lr),
    ('Random Forest (RFE)', y_pred_rfe, y_proba_rfe),
    ('Stacking (RFE)', rfe_stacking_pipeline.predict(X_test), y_proba_rfe_stack),
]

# Calculate comprehensive metrics
metrics_summary = []
for model_name, y_pred, y_proba in all_model_data:
    metrics_summary.append({
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba),
        'Avg Precision': average_precision_score(y_test, y_proba),
        'Brier Score': brier_score_loss(y_test, y_proba)
    })

metrics_df = pd.DataFrame(metrics_summary)

# Save comprehensive metrics
metrics_df.to_csv('Aresults/comprehensive_model_metrics.csv', index=False)
metrics_df.to_csv('Arfe_results/rfe_comparison_metrics.csv', index=False)

# ===============================================================
# 6.1 Performance Heatmap
# ===============================================================
print("Generating performance heatmap...")
fig, ax = plt.subplots(figsize=(12, 8))

# Prepare data for heatmap
heatmap_data = metrics_df.set_index('Model').drop('Brier Score', axis=1)  # Brier score has different scale
heatmap_data = heatmap_data.T  # Transpose so models are columns

# Create heatmap
sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='YlOrRd', 
            linewidths=0.5, linecolor='gray', ax=ax, cbar_kws={'label': 'Score'})

ax.set_title('Model Performance Heatmap (Higher is Better)', fontsize=16, pad=20)
ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Metric', fontsize=12)

plt.tight_layout()
save_figure('model_performance_heatmap', fig)
plt.close(fig)

# ===============================================================
# 6.2 Radar Chart for All Models
# ===============================================================
print("Generating radar chart for all models...")
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))

# Metrics to compare (excluding Brier Score)
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'Avg Precision']
n_metrics = len(metrics)

# Calculate metrics for each model
model_data = {}
for model_name, y_pred, y_proba in all_model_data:
    model_data[model_name] = [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        roc_auc_score(y_test, y_proba),
        average_precision_score(y_test, y_proba)
    ]

# Create angles for radar chart
angles = np.linspace(0, 2 * np.pi, n_metrics, endpoint=False).tolist()
angles += angles[:1]  # Close the loop

# Define colors for different model types
model_colors = {
    'Logistic': '#2E86AB',
    'Random Forest': '#A23B72',
    'Stacking': '#F18F01'
}

# Plot each model
for idx, (model_name, values) in enumerate(model_data.items()):
    # Determine color based on model type
    color_key = model_name.split(' ')[0]  # Get first word
    color = model_colors.get(color_key, colors[idx % len(colors)])
    
    # Determine line style based on feature selection
    linestyle = '-' if '(Full)' in model_name else '--'
    
    values = np.concatenate((values, [values[0]]))  # Close the loop
    ax.plot(angles, values, linestyle, linewidth=2, label=model_name, color=color)
    ax.fill(angles, values, alpha=0.1, color=color)

# Set labels
ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics, fontsize=12)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=10)
ax.set_title('Comprehensive Model Performance Radar Chart', fontsize=16, pad=30)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0), fontsize=11)
ax.grid(True)

save_figure('comprehensive_radar_chart', fig)
plt.close(fig)

# ===============================================================
# 7. FINAL SUMMARY AND SAVING RESULTS
# ===============================================================
print("\n" + "="*50)
print("7. FINAL SUMMARY")
print("="*50)

# Print summary table
print("\n" + "="*80)
print("COMPREHENSIVE MODEL PERFORMANCE SUMMARY")
print("="*80)
print(metrics_df.round(4).to_string())

# Calculate improvement from RFE
print("\n" + "="*80)
print("RFE IMPROVEMENT ANALYSIS")
print("="*80)

for base_model in ['Logistic', 'Random Forest', 'Stacking']:
    full_row = metrics_df[metrics_df['Model'] == f'{base_model} (Full)'].iloc[0]
    rfe_row = metrics_df[metrics_df['Model'] == f'{base_model} (RFE)'].iloc[0]
    
    # Calculate percentage improvement
    auc_improvement = ((rfe_row['ROC-AUC'] - full_row['ROC-AUC']) / full_row['ROC-AUC']) * 100
    f1_improvement = ((rfe_row['F1-Score'] - full_row['F1-Score']) / full_row['F1-Score']) * 100
    
    print(f"\n{base_model}:")
    print(f"  Features reduced from {len(feature_names)} to {best_n_features} ({best_n_features/len(feature_names)*100:.1f}%)")
    print(f"  ROC-AUC: {full_row['ROC-AUC']:.4f} → {rfe_row['ROC-AUC']:.4f} ({auc_improvement:+.2f}%)")
    print(f"  F1-Score: {full_row['F1-Score']:.4f} → {rfe_row['F1-Score']:.4f} ({f1_improvement:+.2f}%)")
    print(f"  Model size reduced (fewer features)")

# Save final summary report
summary_report = f"""
CYBERSECURITY INTRUSION DETECTION PIPELINE - FINAL REPORT
==========================================================

DATA SUMMARY:
- Training samples: {X_train.shape[0]}
- Test samples: {X_test.shape[0]}
- Original features: {len(feature_names)}
- Selected features (RFE): {best_n_features} ({best_n_features/len(feature_names)*100:.1f}% reduction)

RFE OPTIMAL FEATURES: {best_n_features}
Selected Features:
{chr(10).join([f"  {i+1}. {feat}" for i, feat in enumerate(selected_feature_names[:10])])}
{"" if len(selected_feature_names) <= 10 else f"  ... and {len(selected_feature_names)-10} more"}

BEST PERFORMING MODEL:
{metrics_df.loc[metrics_df['ROC-AUC'].idxmax()].to_string()}

CONCLUSIONS:
1. RFE successfully reduced feature space by {100 - (best_n_features/len(feature_names)*100):.1f}%
2. Model performance with RFE-selected features is comparable to full feature set
3. The best model achieves ROC-AUC of {metrics_df['ROC-AUC'].max():.4f}
4. Feature selection improves model interpretability and reduces computational cost
"""

with open('Aresults/final_summary_report.txt', 'w') as f:
    f.write(summary_report)

print("\n" + "="*80)
print("PIPELINE COMPLETED SUCCESSFULLY!")
print("="*80)
print(f"✓ Models saved in: models/")
print(f"✓ Figures saved in: figures/")
print(f"✓ Results saved in: results/")
print(f"✓ RFE results saved in: rfe_results/")
print(f"✓ Optimal number of features selected: {best_n_features}")
print(f"✓ Final summary report: results/final_summary_report.txt")

# Display final metrics table
print("\nFinal Metrics Table:")
print("="*80)
print(metrics_df[['Model', 'Accuracy', 'F1-Score', 'ROC-AUC', 'Avg Precision']].round(4).to_string(index=False))
print("="*80)

print("\nAll visualizations have been saved as PDF and PNG files.")
print("You can find them in the 'figures' directory.")
print("\nThank you for using the Cybersecurity Intrusion Detection Pipeline!")

DATA LOADING & PREPROCESSING
Training set shape: (7629, 9)
Test set shape: (1908, 9)
Class distribution (Train): [4218 3411]
Class distribution (Test): [1055  853]

1. BASELINE LOGISTIC REGRESSION MODEL
Training Baseline Logistic Regression...
✓ Baseline Logistic Regression model saved

2. RANDOM FOREST MODEL
Training Random Forest...
✓ Random Forest model saved

3. HYBRID MODEL (Stacking Classifier)
Training Stacking Classifier...
✓ Stacking model saved

4. RECURSIVE FEATURE ELIMINATION (RFE) ANALYSIS
Number of features after preprocessing: 17

Performing RFE to find optimal number of features...
> 2 features: ROC-AUC = 0.737 (±0.062)
> 3 features: ROC-AUC = 0.850 (±0.038)
> 4 features: ROC-AUC = 0.875 (±0.007)
> 5 features: ROC-AUC = 0.876 (±0.007)
> 6 features: ROC-AUC = 0.887 (±0.006)
> 7 features: ROC-AUC = 0.885 (±0.006)
> 8 features: ROC-AUC = 0.889 (±0.007)
> 9 features: ROC-AUC = 0.888 (±0.006)
> 10 features: ROC-AUC = 0.888 (±0.006)
> 11 features: ROC-AUC = 0.886 (±0.006)
> 1

In [1]:
!pwd

/Users/theonarh/Desktop/attendance/SARKDIST_Projects


Python(66410) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
